In [1]:
from z3 import *
from building_blocks.fin import Fin
from building_blocks.set import *
from building_blocks.mapping import *
from building_blocks.props import *

import z3_monkey
z3_monkey.mk_mapping = MappingSort

In [2]:
from dataclasses import dataclass
from functools import cmp_to_key


@dataclass
class Subdomain:
    sort: SortRef
    univ: SortRef
    proj: ArrayRef  # f : sort -> univ
    inj: BoolRef    # Injective f

    @classmethod
    def mk(cls, name, univ):
        sort = DeclareSort(name)
        f = Const(f"{name}.proj", sort ** univ)
        return cls(sort, univ, f, Injective(f))
        

@dataclass
class ListRef(EvalAt, IGuarded):
    dom: Set          # ι : Set  (index domain)
    ord: ArrayRef     # R : ι -> ι -> B
    val: ArrayRef     # val : ι -> a
    linord: BoolRef   # LinearOrder R

    def guards(self): return [self.linord]

    def eval(self, m: ModelRef):
        return [m.eval(self.val[i]) for i in ordered(self.dom, self.ord, m)]
        

def List(name, value_sort):
    I = Set(f"{name}.ι", U)
    R = Mapping(f"{name}.ord", I ** I ** bool)
    f = Mapping(f"{name}.val", I ** value_sort)
    return ListRef(I, R, f, LinearOrder(R))


def comparator(R: dict | MappingRef, m: ModelRef=None):
    ev = m.eval if m else lambda x: x
    def cmp(x, y):
        if x == y: return 0
        if ev(R[x][y]): return -1
        return 1
    return cmp

def ordered(l: list | set | SetRef, R: dict | MappingRef, m: ModelRef=None):
    if isinstance(l, SetRef): l = l @ m
    return sorted(l, key=cmp_to_key(comparator(R, m)))


#U = DeclareSort("○")
U = Fin[3]
A = DeclareSort("α")


l1 = List("l₁", A)
l1

ListRef(dom=l₁.ι, ord=l₁.ord, val=l₁.val, linord=And(And(And(ForAll(x, Implies(l₁.ι[x], l₁.ord[x][x])),
            ForAll([x, y, z],
                   Implies(l₁.ι[x],
                           Implies(l₁.ι[y],
                                   Implies(l₁.ι[z],
                                        Implies(l₁.ord[x][y],
                                        Implies(l₁.ord[y][z],
                                        l₁.ord[x][z]))))))),
        ForAll([x, y],
               Implies(l₁.ι[x],
                       Implies(l₁.ι[y],
                               Implies(l₁.ord[x][y],
                                       Implies(l₁.ord[y][x],
                                        x == y)))))),
    ForAll([x, y],
           Implies(l₁.ι[x],
                   Implies(l₁.ι[y],
                           Or(l₁.ord[x][y], l₁.ord[y][x]))))))

In [3]:

def sublist(l1, l2):
    i1, i2 = l1.dom, l2.dom
    R1, R2 = l1.ord, l2.ord
    f = Mapping('f', i1 ** i2)
    x, y = Elements('x y', i1)
    return Exists([f], f.guards() & Injective(f) &
                  ForAll([x, y], R1[x][y] == R2[f[x]][f[y]]) &
                  ForAll([x], l1.val[x] == l2.val[f[x]]))

l2 = List("l₂", A)
sublist(l1, l2)

Exists(f,
       And(ForAll(x,
                  Implies(l₁.ι[x],
                          Exists(y, And(l₂.ι[y], f[x] == y)))),
           And(And(And(ForAll(x,
                              Implies(l₁.ι[x],
                                      Exists(y,
                                        And(l₂.ι[y],
                                        f[x] == y)))),
                       ForAll([x, y],
                              Implies(l₁.ι[x],
                                      Implies(l₁.ι[y],
                                        Implies(f[x] == f[y],
                                        x == y))))),
                   ForAll([x, y],
                          Implies(l₁.ι[x],
                                  Implies(l₁.ι[y],
                                        l₁.ord[x][y] ==
                                        l₂.ord[f[x]][f[y]])))),
               ForAll(x,
                      Implies(l₁.ι[x],
                              l₁.val[x] == l₂.val[f[x]])))))

In [4]:
l3 = List("l₃", A)

s31 = Set("S", U)

s = Solver()

s.add(*l1.guards(), *l2.guards(), *l3.guards())
s.add(subset(s31, l3.dom))

f1 = Mapping('f1', l1.dom ** s31)
f2 = Mapping('f2', l2.dom ** set_diff(l3.dom, s31))

s.add(*f1.guards(), *f2.guards(),
    Isomorphism(f1) & Prod([f1.domain()], lambda i: l1.val[i] == l3.val[f1[i]]),
    Isomorphism(f2) & Prod([f2.domain()], lambda i: l2.val[i] == l3.val[f2[i]]),
    Prod([f1.domain(), f2.domain()], lambda i, j: l3.ord[f1[i]][f2[j]]),
    Injective(l3.val))

#s.add(Isomorphism(s31, l1.dom) & isomorphic(set_diff(l3.dom, s31), l2.dom))

s.check()

s.add(card_geq(l1.dom, 1))
s.add(card_geq(l2.dom, 2))

m = s.model() if (r := s.check()) == sat else r
m

[l₁.ι = Store(Store(K(○, True), ⓪, False), ①, False),
 l₃.ι = K(○, True),
 f2 = Store(K(○, ⓪), ⓪, ①),
 l₃.val = Lambda(k!0,
                 If(If(k!0 ==
                       If(Or(And(Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ①,
                                       False) ==
                                 Store(K(○, False), ②, True),
                                 Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ①,
                                       False) ==
                                 Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ②,
                                       False)),
                             And(Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ①,
                                       False) ==
                                 Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                       ①,
                                       True),
                                 Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ①,
                                       False) ==
                                 K(○, True))),
                          ①,
                          ⓪),
                       False,
                       If(k!0 ==
                          If(Or(And(Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                        ①,
                                        False) ==
                                    Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                        ①,
                                        True),
                                    Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                        ①,
                                        True) ==
                                    K(○, True)),
                                And(Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                        ①,
                                        False) ==
                                    Store(K(○, False),
                                        ②,
                                        True),
                                    Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                        ①,
                                        True) ==
                                    Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                        ②,
                                        False))),
                             ②,
                             ⓪),
                          Or(And(Store(Store(K(○, True),
                                        ⓪,
                                        False),
                                       ①,
                                       False) ==
                                 Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                       ①,
                                      

In [5]:
(ordered(l1.dom, l1.ord, m), l1 @ m), \
(ordered(l2.dom, l2.ord, m), l2 @ m), (ordered(l3.dom, l2.ord, m), l3 @ m), f1 @ m, f2 @ m

(([②], [α!val!0]),
 ([⓪, ①], [α!val!1, α!val!2]),
 ([⓪, ②, ①], [α!val!0, α!val!1, α!val!2]),
 {②: ②},
 {⓪: ①, ①: ⓪})

In [6]:
#from building_blocks.mapping import tabulate
from building_blocks.formula_manip import alpha_renaming, RenamingVisitor

f_tbl = Mapping('f', l2.dom ** l1.dom)


s = Solver()
s.set('timeout', 5000)
s.add(*l1.guards(), *l2.guards(), sublist(l1, l2))#, Not(sublist(l2, l1)))
#s.add(sublist(l2, l1))
    
s.add(card_geq(l1.dom, 1))
s.add(card_geq(l2.dom, 2))

s31 = Set("S", U)

s.add(*l3.guards())
s.add(subset(s31, l3.dom))
s.add(isomorphic(s31, l1.dom) & isomorphic(set_diff(l3.dom, s31), l2.dom))


if 0:
    
    
    f_tbl = tabulate(f_tbl)
    f_cexs = [
        tabulate(f_tbl, table={U[0]: U[1], U[2]: U[2], U[1]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[1], U[1]: U[2], U[2]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[2], U[1]: U[0], U[2]: U[1]}),
        tabulate(f_tbl, table={U[0]: U[2], U[1]: U[1], U[2]: U[0]}),
        tabulate(f_tbl, table={U[0]: U[0], U[1]: U[1], U[2]: U[2]}),
        tabulate(f_tbl, table={U[0]: U[0], U[1]: U[2], U[2]: U[1]})
    ]
    
    s.add(*(Not(
        substitute_vars(sublist(l2, l1).body(), f_tbl)) for f_tbl in f_cexs))
    #    Exists(list(f_tbl.table.values()), substitute_vars(sublist(l2, l1).body(), f_tbl))))


s.add(disjoint(l1.dom, l2.dom))

m = s.model() if (r := s.check()) == sat else r
m

[l₁.ι = Store(K(○, False), ②, True),
 l₃.ι = Store(Store(Store(K(○, False), ⓪, True), ①, True),
              ②,
              True),
 l₂.val = K(○, α!val!0),
 l₁.ord = Store(Store(K(○,
                        Store(Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                    ①,
                                    True),
                              ②,
                              True)),
                      ⓪,
                      K(○, False)),
                ②,
                K(○, True)),
 l₂.ord = Store(Store(K(○,
                        Store(Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                    ①,
                                    True),
                              ②,
                              True)),
                      ⓪,
                      Store(K(○, True), ①, False)),
                ①,
                Store(Store(K(○, False), ⓪, True), ①, True)),
 l₂.ι = Store(K(○, True), ②, False),
 l₃.ord = Store(Store(K(○,
                        Store(Store(Store(K(○, False),
                                        ⓪,
                                        True),
                                    ①,
                                    True),
                              ②,
                              True)),
                      ⓪,
                      Store(K(○, True), ②, False)),
                ①,
                Store(Store(K(○, True), ⓪, False), ②, False)),
 l₁.val = Lambda(k!0,
                 If(If(k!0 ==
                       If(Store(Store(K(○, ⓪), ②, ①), ①, ②)[If(And(Store(K(○,
                                        False),
                                        ②,
                                        True) ==
                                   Store(K(○, True),
                                        ②,
                                        False),
                                   Store(K(○, False),
                                        ①,
                                        True) ==
                                   Store(Store(Store(K(○,
                                        False),
                                        ⓪,
                                        True),
                                        ①,
                                        True),
                                        ②,
                                        True)),
                               ⓪,
                               ②)] ==
                          ①,
                          ②,
                          ⓪),
                       Store(Store(K(○, ⓪), ②, ①), ①, ②)[If(And(Store(K(○,
                                        False),
                                      ②,
                                      True) ==
                                Store(K(○, True), ②, False),
                                Store(K(○, False), ①, True) ==
                                Store(Store(Store(K(○,
                                        False),
                                        ⓪,
                                        True),
                                        ①,
                                        True),
                                      ②,
                                      True)),
                            ⓪,
                            ②)] ==
                       ①,
                       If(k!0 ==
                          If(Store(Store(K(○, ⓪), ②, ①),
                                   ①,
                                   ②)[If(And(Store(K(○,
                                        False),
                                        ②,
                                        True) ==
                                      Store(K(○, True),
                                        ②,
                                        False),
                                      

In [7]:
#m.eval(l1.dom.set), m.eval(l2.dom.set), m.eval(f_id)

ordered(l1.dom, l1.ord, m), ordered(l2.dom, l2.ord, m), (f_tbl @ m if f_tbl in m.decls() else {}), \
 l1 @ m, l2 @ m, ordered(l3.dom, l3.ord, m)

([②], [①, ⓪], {}, [α!val!0], [α!val!0, α!val!0], [②, ⓪, ①])

In [8]:
diag = [
    d() == m.get_interp(d) for d in m.decls() if d.arity() == 0]

diag
s_aux = Solver()
s_aux.add(*diag)
s_aux.add(sublist(l2, l1))
s_aux.check()

unsat

In [9]:
with open('outf.smt2', 'w') as outf: outf.write(s.to_smt2())

In [10]:
U.values()

[⓪, ①, ②]

In [11]:
U[0]

⓪